In [3]:
import polars as pl
from datetime import datetime
from datetime import date, time
import glob
import os
from collections import defaultdict

In [4]:
first_glob = os.path.expanduser("~").replace("\\", "/")

exclibur_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Performance All Sites.xlsx'
output_req = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Req.csv'
parquet_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/excalibur_raw.parquet'

df = pl.read_excel(
    exclibur_path,
    sheet_name="Requierd Hours",
    engine="calamine",
    has_header=False,
)

# Deduplicate headers
raw_headers = [str(val).split(" ")[0] for val in df.row(0)]
seen = defaultdict(int)
new_headers = []
for h in raw_headers:
    count = seen[h]
    new_headers.append(h if count == 0 else f"{h}_{count}")
    seen[h] += 1

mtd_index = next((i for i, h in enumerate(new_headers) if h == "MTD"), None)
if mtd_index is None:
    raise ValueError(f"Không tìm thấy cột MTD! Headers hiện tại: {new_headers}")

cols_to_keep = df.columns[:mtd_index + 1]
rename_mapping = dict(zip(cols_to_keep, new_headers[:mtd_index + 1]))

result_df = df.slice(1, 980).select(cols_to_keep).rename(rename_mapping)

result_df = result_df.with_columns([
    pl.col("PSP").replace_strict(
        {"Chat Non-Lodging": "Non-Lodging chat"},
        default=pl.col("PSP")
    ),
    pl.col("Interval")
      .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
      .dt.strftime("%H:%M:%S"),
])

data_cols = [c for c in result_df.columns if c not in ("PSP", "Site", "Interval")]

def to_numeric_expr(col_name: str, dtype) -> pl.Expr:
    dtype_str = str(dtype)
    if dtype_str.startswith("Datetime") or dtype_str.startswith("Time"):
        return (
            pl.col(col_name).dt.hour()
            + pl.col(col_name).dt.minute() / 60
            + pl.col(col_name).dt.second() / 3600
            + pl.col(col_name).dt.microsecond() / 3_600_000_000
        ).alias(col_name)
    else:
        return pl.col(col_name).cast(pl.Float64, strict=False).alias(col_name)

result_df = result_df.with_columns([
    to_numeric_expr(c, result_df.schema[c]) for c in data_cols
])

result_df = result_df.filter(
    pl.col("PSP").is_in(["Lodging chat", "Non-Lodging chat"])
).sort("PSP", maintain_order=True)

result_df.write_csv(output_req, include_bom=True)
print(result_df["PSP"].unique(maintain_order=True))
result_df.head(10)

shape: (2,)
Series: 'PSP' [str]
[
	"Lodging chat"
	"Non-Lodging chat"
]


PSP,Site,Interval,6/29/2026,6/30/2026,7/1/2026,7/2/2026,7/3/2026,7/4/2026,7/5/2026,7/6/2026,7/7/2026,7/8/2026,7/9/2026,7/10/2026,7/11/2026,7/12/2026,7/13/2026,7/14/2026,7/15/2026,7/16/2026,7/17/2026,7/18/2026,7/19/2026,7/20/2026,7/21/2026,7/22/2026,7/23/2026,7/24/2026,7/25/2026,7/26/2026,7/27/2026,7/28/2026,7/29/2026,7/30/2026,7/31/2026,MTD
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lodging chat""","""Concentrix (Pune)""","""00:00:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""00:30:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""01:00:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""01:30:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""02:00:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""02:30:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""03:00:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""03:30:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
"""Lodging chat""","""Concentrix (Pune)""","""04:00:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0


In [13]:
def process_psp_hours(file_path):
    try:
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)
        
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime) else str(val).strip() if val is not None else "Unknown" 
            for val in header_vals
        ]
        df_raw.columns = new_columns
        
        df = df_raw.slice(1).with_columns(
            pl.col("Interval").cast(pl.String).str.replace(r"^.*1899-12-31\s+", "").str.slice(0, 8).alias("Interval")
        )
        
        final_df = (df.unpivot(index=["LOB", "Site", "Interval"], variable_name="Date_Str", value_name="Value")
            .with_columns([
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                .dt.truncate("1m").alias("PST_Datetime"),
                
                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("PSP"),
                pl.col("LOB").str.strip_chars(), 
                pl.col("Site").str.strip_chars()
            ])
            .with_columns(
                pl.col("PSP").sum().over(["LOB", "PST_Datetime"]).alias("Total PSP")
            )
        )
        return final_df.select(["LOB", "Site", "PST_Datetime", "PSP", "Total PSP"])
    except Exception:
        return pl.DataFrame()

def process_productive_hours(folder_path):
    file_pattern = os.path.join(folder_path, "*.csv")
    all_files = sorted(glob.glob(file_pattern))
    
    dfs = []
    
    for file in all_files:
        df_temp = pl.read_csv(
            file,
            infer_schema_length=0,
            ignore_errors=True,
            encoding="utf-8",
        )
        
        rename_map = {
            "Forecast Group Name": "LOB",
            "Business Location": "Site",
            "Interval": "Raw_Interval",
            "Productive Hours (Sum)": "Productive Hours"
        }
        existing_rename = {k: v for k, v in rename_map.items() if k in df_temp.columns}
        if existing_rename:
            df_temp = df_temp.rename(existing_rename)
        
        if "Productive Hours" in df_temp.columns:
            df_temp = df_temp.with_columns(
                pl.col("Productive Hours")
                .cast(pl.String)
                .str.strip_chars()
                .str.replace_all(r",", "")
                .str.replace_all(r"[^\d.-]", "")
                .replace({"": "0", "-": "0", "N/A": "0", "null": "0"})
                .cast(pl.Float64, strict=False)
                .fill_null(0.0)
                .alias("Productive Hours")
            )
        
        for col in ["LOB", "Site", "Raw_Interval"]:
            if col in df_temp.columns:
                df_temp = df_temp.with_columns(pl.col(col).cast(pl.String).str.strip_chars())
        
        dfs.append(df_temp)
    
    df_raw = pl.concat(dfs, how="vertical_relaxed")
    
    df = df_raw.select([
        pl.col("LOB"),
        pl.col("Site"),
        pl.col("Raw_Interval"),
        pl.col("Productive Hours")
    ])
    
    df = df.with_columns([
        pl.when(pl.col("LOB") == "GEN_GEN_EN_GCS_GLG_CHT")
        .then(pl.lit("Lodging chat"))
        .when(pl.col("LOB") == "GEN_GEN_EN_GCS_GNL_CHT")
        .then(pl.lit("Non-Lodging chat"))
        .otherwise(pl.col("LOB"))
        .str.strip_chars()
        .alias("LOB"),
        pl.col("Site").str.strip_chars().alias("Site")
    ])

    df_agg = df.group_by(["LOB", "Site", "Raw_Interval"]).agg(
        pl.col("Productive Hours").sum()
    )

    df_processed = df_agg.with_columns(
        pl.col("Raw_Interval")
          .str.strip_chars()
          .str.to_datetime(format="%m/%d/%Y %H:%M", strict=False)
          .fill_null(
              pl.col("Raw_Interval")
                .str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False)
          )
          .dt.truncate("1m")
          .alias("PST_Datetime")
    ).filter(
        pl.col("PST_Datetime").is_not_null()
    ).with_columns(
        pl.col("Productive Hours")
          .sum()
          .over(["LOB", "PST_Datetime"])
          .alias("Total Productive Hours")
    )

    return df_processed.select([
        "LOB", "Site", "PST_Datetime", "Productive Hours", "Total Productive Hours"
    ])

def process_vendor_msp(folder_path: str) -> pl.DataFrame:
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    if not files:
        return pl.DataFrame()

    target_columns = [
        "Vendor Forecast Group Name",
        "Interval Time",
        "Forecast Productive Hour (Sum)",
        "Occupancy",
        "Staffing Attainment (Pct)",
        "Interval Compliance (Pct)"
    ]
    
    schema_overrides = {
        "Forecast Productive Hour (Sum)": pl.Float64,
        "Occupancy": pl.Float64,
        "Staffing Attainment (Pct)": pl.Float64,
        "Interval Compliance (Pct)": pl.Float64,
        "Interval Time": pl.String
    }

    dfs = []
    
    for f in files:
        try:
            d = pl.read_csv(
                f,
                columns=target_columns,
                schema_overrides=schema_overrides,
                infer_schema_length=0
            )
            dfs.append(d)
        except Exception:
            continue

    if not dfs:
        return pl.DataFrame()

    df = pl.concat(dfs, how="vertical")

    df_processed = df.with_columns([
        pl.when(pl.col("Vendor Forecast Group Name") == "GEN_GEN_EN_GCS_GLG_CHT_Concentrix")
          .then(pl.lit("Lodging chat"))
          .when(pl.col("Vendor Forecast Group Name") == "GEN_GEN_EN_GCS_GNL_CHT_Concentrix")
          .then(pl.lit("Non-Lodging chat"))
          .otherwise(pl.col("Vendor Forecast Group Name"))
          .alias("LOB"),

        pl.coalesce([
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%m/%d/%Y %H:%M", strict=False),
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%m/%d/%Y %I:%M:%S %p", strict=False)
        ]).alias("PST_Datetime"),

        pl.col("Forecast Productive Hour (Sum)").alias("MSP"),
        pl.col("Occupancy"),
        pl.col("Staffing Attainment (Pct)"),
        pl.col("Interval Compliance (Pct)")
    ])

    return df_processed.select([
        "LOB", 
        "PST_Datetime", 
        "MSP", 
        "Occupancy", 
        "Staffing Attainment (Pct)", 
        "Interval Compliance (Pct)"
    ])

def calculate_msp_distribution(df):
    df = df.with_columns([
        pl.col("Productive Hours").cast(pl.Float64).fill_null(0.0),
        pl.col("Total Productive Hours").cast(pl.Float64).fill_null(0.0),
        pl.col("MSP").cast(pl.Float64).fill_null(0.0).round(4),
        pl.col("PSP").cast(pl.Float64).fill_null(0.0).round(4),
        pl.col("Total PSP").cast(pl.Float64).fill_null(0.0).round(4),
        
        (pl.col("PST_Datetime").dt.strftime("%H:%M") + "-" +
         (pl.col("PST_Datetime") + pl.duration(minutes=29)).dt.strftime("%H:%M")).alias("PST_Interval_Range"),
        
        pl.col("PST_Datetime").dt.time().alias("PST_Interval"),
        pl.col("PST_Datetime").dt.date().alias("Date"),
        
        (pl.col("PST_Datetime")
         .dt.replace_time_zone("America/Los_Angeles", ambiguous="earliest", non_existent="null")
         .dt.convert_time_zone("Asia/Ho_Chi_Minh")
         .dt.replace_time_zone(None)
         .alias("VNT_Datetime")
        )
    ])
    
    df = df.with_columns([
        (pl.col("MSP") - pl.col("Total PSP")).round(4).alias("Variance (MSP-PSP)"),
        
        pl.when(pl.col("Site").str.contains("Ho Chi Minh"))
          .then(pl.col("PSP"))
          .otherwise(0.0)
          .sum().over(["LOB", "PST_Datetime"])
          .round(4).alias("PSP_HCM")
    ]).with_columns(
        (pl.col("Total PSP") - pl.col("PSP_HCM")).round(4).alias("Total_PSP_Non_HCM")
    )

    is_lodging = pl.col("LOB").str.to_lowercase().str.contains("lodging chat")
    is_hcm = pl.col("Site").str.contains("Ho Chi Minh")
    is_single_site = (pl.col("PSP") - pl.col("Total PSP")).abs() < 0.0001
    is_positive_var = pl.col("Variance (MSP-PSP)") > 0

    calc_prorated = pl.col("Variance (MSP-PSP)") * (pl.col("PSP") / pl.col("Total PSP"))
    calc_loss_sharing = pl.col("Variance (MSP-PSP)") * (pl.col("PSP") / pl.col("Total_PSP_Non_HCM").replace(0, 1))

    df = df.with_columns(
        pl.when(
            is_lodging & (is_positive_var | is_single_site)
        )
        .then(calc_prorated) 
        .otherwise(
            pl.when(is_lodging & is_hcm)
            .then(0.0)
            .otherwise(
                pl.when(is_lodging)
                .then(calc_loss_sharing)
                .otherwise(calc_prorated)
            )
        )
        .fill_nan(0.0).fill_null(0.0).round(4)
        .alias("Initial MSP allocation")
    )

    df = df.with_columns(
        pl.when((pl.col("PSP") + pl.col("Initial MSP allocation")) < -0.0001)
        .then(-pl.col("PSP"))
        .otherwise(pl.col("Initial MSP allocation"))
        .alias("Initial Adjustment")
    ).with_columns(
        (pl.col("Initial MSP allocation") - pl.col("Initial Adjustment")).alias("Pending MSP to be assigned")
    ).with_columns(
        pl.col("Pending MSP to be assigned").sum().over(["LOB", "PST_Datetime"]).alias("Total_Pending_Global")
    ).with_columns(
        pl.when(is_lodging & pl.col("Site").str.contains("Ho Chi Minh"))
        .then(pl.col("Initial Adjustment") + pl.col("Total_Pending_Global"))
        .otherwise(pl.col("Initial Adjustment"))
        .alias("Final adjustment")
    )

    return df.with_columns(
        (pl.col("PSP") + pl.col("Final adjustment")).alias("MSP site wise")
    )

def calculate_billable_logic(df):
    return (df
        .with_columns([
            pl.min_horizontal(["MSP site wise", "Productive Hours"]).alias("Billable"),
            pl.when(pl.col("Productive Hours") >= pl.col("MSP site wise"))
            .then(0.0).otherwise(pl.col("MSP site wise") - pl.col("Productive Hours")).alias("Billable Loss"),
            pl.when((pl.col("Productive Hours") - pl.col("MSP site wise")) < 0)
            .then(0.0).otherwise(pl.col("Productive Hours") - pl.col("MSP site wise")).alias("Over Production")
        ])
        .with_columns([
            pl.col("Billable Loss").sum().over(["LOB", "PST_Datetime"]).alias("Global Billable Loss"),
            pl.col("Over Production").sum().over(["LOB", "PST_Datetime"]).alias("Global Over Production")
        ])
        .with_columns(
            pl.when(pl.col("Billable Loss") > 0).then(0.0)
            .otherwise((pl.col("Global Billable Loss") * pl.col("Over Production")) / pl.col("Global Over Production"))
            .fill_nan(0.0).fill_null(0.0).alias("Compensated by other site")
        )
        .with_columns([
            pl.min_horizontal(["Over Production", "Compensated by other site"]).alias("Actual Compensation"),
            (pl.col("Billable") + pl.min_horizontal(["Over Production", "Compensated by other site"])).alias("Final Billable")
        ])
        .with_columns((pl.col("Final Billable") - pl.col("MSP site wise")).alias("Unbillable Hours"))
    )

def run_pipeline():
    req_path  = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OU.xlsx")
    prod_path = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_PRODUCTIVE_REVENUE")
    wfm_path  = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_WORKFORCE_VENDOR")

    df_req  = process_psp_hours(req_path)
    df_prod = process_productive_hours(prod_path)
    df_msp  = process_vendor_msp(wfm_path)

    if df_req.is_empty() or df_prod.is_empty() or df_msp.is_empty():
        return None

    df_merged = (
        df_req
        .join(df_prod, on=["LOB", "Site", "PST_Datetime"], how="left", coalesce=True)
        .join(df_msp,  on=["LOB", "PST_Datetime"],         how="left", coalesce=True)
    )

    df_distributed = calculate_msp_distribution(df_merged)
    df_final       = calculate_billable_logic(df_distributed)

    final_cols = [
        "LOB", "Site", "Date", "PST_Interval", "PST_Interval_Range", "VNT_Interval", "VNT_Interval_Range", "PST_Datetime", "VNT_Datetime",
        "PSP", "Total PSP", "MSP", "Variance (MSP-PSP)", "PSP_HCM", "Total_PSP_Non_HCM",
        "Initial MSP allocation", "Initial Adjustment", "Pending MSP to be assigned", "Final adjustment",
        "MSP site wise", "Productive Hours", "Total Productive Hours", "Billable", "Billable Loss", "Over Production",
        "Compensated by other site", "Actual Compensation", "Final Billable", "Unbillable Hours",
        "Occupancy", "Staffing Attainment (Pct)", "Interval Compliance (Pct)"
    ]

    existing_cols = [c for c in final_cols if c in df_final.columns]
    df_result = df_final.select(existing_cols).sort(["LOB", "Site", "PST_Datetime"])

    df_result.write_excel(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/excalibur_raw.xlsx')
    df_result.write_parquet(parquet_path)

    return df_result

df_result = run_pipeline()

In [14]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)

target_cols = [
    "PST_Interval", "PSP", "Total PSP", "MSP", 
    "Variance (MSP-PSP)", "PSP_HCM", "Total_PSP_Non_HCM",
    "Initial MSP allocation", 
    "Initial Adjustment", 
    "Pending MSP to be assigned", 
    "Final adjustment", 
    "MSP site wise"
]

df_result.filter(
    (pl.col("Date") == date(2026, 6, 10)) & 
    (pl.col("PST_Interval") >= time(13, 0, 0)) &
    (pl.col("Site") == "Concentrix (Ho Chi Minh City)")
).select(target_cols).sort("PST_Interval")

PST_Interval,PSP,Total PSP,MSP,Variance (MSP-PSP),PSP_HCM,Total_PSP_Non_HCM,Initial MSP allocation,Initial Adjustment,Pending MSP to be assigned,Final adjustment,MSP site wise
time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
13:00:00,9.155,18.73,18.7289,-0.0011,9.155,9.575,0.0,0.0,0.0,0.0,9.155
13:00:00,1.16,10.47,10.9868,0.5168,1.16,9.31,0.0573,0.0573,0.0,0.0573,1.2173
13:30:00,9.275,18.73,18.7289,-0.0011,9.275,9.455,0.0,0.0,0.0,0.0,9.275
13:30:00,0.69,9.065,9.4935,0.4285,0.69,8.375,0.0326,0.0326,0.0,0.0326,0.7226
14:00:00,9.55,16.21,16.208,-0.002,9.55,6.66,0.0,0.0,0.0,0.0,9.55
14:00:00,1.47,9.355,9.8035,0.4485,1.47,7.885,0.0705,0.0705,0.0,0.0705,1.5405
14:30:00,10.275,16.2,16.1997,-0.0003,10.275,5.925,0.0,0.0,0.0,0.0,10.275
14:30:00,4.665,8.24,8.6301,0.3901,4.665,3.575,0.2209,0.2209,0.0,0.2209,4.8859
15:00:00,10.4,17.215,17.2164,0.0014,10.4,6.815,0.0008,0.0008,0.0,0.0008,10.4008


In [15]:
import polars as pl

def view_hcm_daily_summary(df):
    print("🚀 TỔNG HỢP SỐ LIỆU NGÀY CHO SITE: Concentrix (Ho Chi Minh City)")
    
    # 1. Lọc Site HCM
    df_hcm = df.filter(pl.col("Site").str.contains("Ho Chi Minh"))
    
    if df_hcm.is_empty():
        print("❌ Không tìm thấy dữ liệu của Site HCM!")
        return

    # 2. Cộng dồn theo Ngày (Group by Date)
    # Các cột cần tính tổng
    target_cols = ["PSP", "MSP", "MSP site wise", "Final Billable", "Billable Loss"]
    # Chỉ lấy các cột thực sự có trong DataFrame
    valid_cols = [c for c in target_cols if c in df_hcm.columns]

    df_daily = df_hcm.group_by("Date").agg(
        [pl.col(c).sum() for c in valid_cols]
    ).sort("Date")

    # 3. Xoay bảng (Pivot) để hiển thị giống Excel
    # Dòng là Tên Chỉ Số, Cột là Ngày
    
    # Bước A: Unpivot (Chuyển cột thành dòng dữ liệu)
    df_long = df_daily.unpivot(
        index="Date", 
        variable_name="Metric", 
        value_name="Value"
    )
    
    # Bước B: Tạo cột ngày dạng chuỗi
    df_ready = df_long.with_columns(
        pl.col("Date").dt.strftime("%Y-%m-%d").alias("Date_Str")
    )

    # Bước C: Pivot (Metric làm dòng, Date làm cột)
    df_pivot = df_ready.pivot(
        values="Value",
        index="Metric",
        on="Date_Str",
        aggregate_function="sum",
        sort_columns=True
    )

    # 4. In kết quả
    print("\n📊 BẢNG TỔNG HỢP NGÀY (Đơn vị: Giờ)")
    # Cấu hình in rộng để thấy hết các ngày
    with pl.Config(tbl_rows=20, tbl_cols=20, float_precision=2, tbl_width_chars=300):
        print(df_pivot)

# =============================================================================
# CÁCH CHẠY:
# =============================================================================
if 'df_final' in locals():
    view_hcm_daily_summary(df_final)
elif 'df_result' in locals():
    view_hcm_daily_summary(df_result)
else:
    print("⚠️ Chưa có biến kết quả df_final/df_result. Hãy chạy code tính toán chính trước!")

🚀 TỔNG HỢP SỐ LIỆU NGÀY CHO SITE: Concentrix (Ho Chi Minh City)

📊 BẢNG TỔNG HỢP NGÀY (Đơn vị: Giờ)
shape: (5, 239)
┌────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ Metric         ┆ 2025-12-29 ┆ 2025-12-30 ┆ 2025-12-31 ┆ 2026-01-01 ┆ 2026-01-02 ┆ 2026-01-03 ┆ 2026-01-04 ┆ 2026-01-05 ┆ 2026-01-06 ┆ … ┆ 2026-08-14 ┆ 2026-08-15 ┆ 2026-08-16 ┆ 2026-08-17 ┆ 2026-08-18 ┆ 2026-08-19 ┆ 2026-08-20 ┆ 2026-08-21 ┆ 2026-08-22 ┆ 2026-08-23 │
│ ---            ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ str            ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64